<a href="https://colab.research.google.com/github/gustavomqv/atividade_py_spark/blob/main/Atividade_Pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Atividade PySpark
Aluno: Gustavo Miguel Queiroz Viana
Matrícula: 2422120014

##Preparação do ambiente

In [8]:
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/nyc_tripdata_2024_sample_4M.csv

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (SparkSession.builder
         .appName("ExerciciosPySpark")
         .master("local[*]")
         .getOrCreate()
)

df = spark.read.csv("nyc_tripdata_2024_sample_4M.csv", header=True,
                    inferSchema=True)

##Questão 1)
Depois de carregar o DataFrame df, exiba:

a) o schema inferido (tipos de cada coluna);

b) as 10 primeiras linhas;

c) o número total de linhas do dataset.

In [2]:
#a)
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [3]:
#b)
df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-10-01 00:59:55|  2024-10-01 01:02:24|              1|          0.5|         1|                 N|         230|         161|           1|        5.1|  3.5|    0.5|       2.

In [4]:
#c)
total_de_linhas = df.count()
print(f"Número total de linhas: {total_de_linhas}")

Número total de linhas: 4118743


##Questão 2)
Selecione apenas as colunas VendorID, tpep_pickup_datetime, trip_distance, fare_amount e payment_type, e exiba as 5 primeiras linhas do resultado.

In [5]:
df_selecionado = df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "payment_type"
)

df_selecionado.show(5)

+--------+--------------------+-------------+-----------+------------+
|VendorID|tpep_pickup_datetime|trip_distance|fare_amount|payment_type|
+--------+--------------------+-------------+-----------+------------+
|       1| 2024-10-01 00:59:55|          0.5|        5.1|           1|
|       1| 2024-10-01 00:08:59|         20.6|       76.5|           2|
|       2| 2024-10-01 00:18:38|         7.42|       33.1|           4|
|       2| 2024-10-01 00:20:06|        19.96|       70.0|           1|
|       1| 2024-10-01 00:09:02|          2.6|       15.6|           1|
+--------+--------------------+-------------+-----------+------------+
only showing top 5 rows


## Questão 3)
Filtre as corridas que atendem simultaneamente às duas condições abaixo, e exiba quantas corridas
restaram:

trip_distance maior que 5 milhas;

passenger_count maior ou igual a 3.

In [9]:
df_filtrado = df.filter(
    (F.col("trip_distance") > 5) & (F.col("passenger_count") >= 3)
)

print(f"Quantidade de corridas que restaram: {df_filtrado.count()}")

Quantidade de corridas que restaram: 50665


##Questão 4)
No comando de preparação deste exercício, o DataFrame foi carregado com inferSchema=True, deixando o
Spark descobrir sozinho o tipo de cada coluna. Explique como esse processo de inferência funciona, e
compare com a alternativa de definir o schema manualmente usando StructType/StructField (como
fizemos no notebook do Colab da aula anterior). Quais são as vantagens e os riscos de cada abordagem,
especialmente pensando num arquivo com milhões de linhas como o deste exercício?

RESPOSTA: Usar o inferschema=True facilita bastante o exercício, porque o spark identifica sozinho os tipos de cada coluna sem precisar definir tudo manualmente. O problema é que o arquivo tendo 4 milhões de linhas, essa etapa pode deixar o carregamento mais demorado. Já ao criar o schema manualmente com o structtype, os tipos das colunas são informados diretamente ao Spark. Mesmo  dando mais de trabalho para montar o código, criar o schema manualmente deixa o carregamento mais rápido e também dimnui erro na hora de identificar os tipos de dados.

##Questão 5)
Agrupe as corridas por payment_type e calcule, para cada grupo:

a quantidade de corridas;

a soma total de total_amount (receita total).

Exiba o resultado ordenado pela receita total, da maior para a menor.

In [10]:
df_agrupado = df.groupBy("payment_type") \
    .agg(
        F.count("*").alias("quantidade_corridas"),
        F.sum("total_amount").alias("receita_total")
    ) \
    .orderBy(F.desc("receita_total"))

df_agrupado.show()

+------------+-------------------+--------------------+
|payment_type|quantidade_corridas|       receita_total|
+------------+-------------------+--------------------+
|           1|            3045849| 9.116799616010016E7|
|           2|             553536|1.2987084559999354E7|
|           0|             410746|1.0123049400000528E7|
|           3|              29100|  220775.24999999974|
|           4|              79511|  133192.01999999984|
|           5|                  1|                62.0|
+------------+-------------------+--------------------+



##Questão 6)
Crie uma nova coluna chamada hora_embarque, extraindo apenas a hora (0 a 23) da coluna
tpep_pickup_datetime. Em seguida, agrupe por hora_embarque e calcule a tarifa média (fare_amount) e a
distância média (trip_distance) para cada hora do dia.

Exiba o resultado ordenado pela hora, de 0 a 23.

In [11]:
df_hora_embarque = df.withColumn("hora_embarque", F.hour("tpep_pickup_datetime")) \
    .groupBy("hora_embarque") \
    .agg(
        F.avg("fare_amount").alias("tarifa_media"),
        F.avg("trip_distance").alias("distancia_media")
    ) \
    .orderBy("hora_embarque")

df_hora_embarque.show(24)

+-------------+------------------+------------------+
|hora_embarque|      tarifa_media|   distancia_media|
+-------------+------------------+------------------+
|            0| 19.72867660335703| 5.130178643081671|
|            1| 17.54847894641617| 3.739999483030465|
|            2|16.426538461538446| 4.542445678033303|
|            3| 17.24064640950263| 3.401675265462839|
|            4| 22.33585451861572|11.412191651631977|
|            5|26.226564065583663| 23.33988120540463|
|            6|21.931859821807333|14.540589393296582|
|            7| 19.33026953083623|11.087329050022918|
|            8|18.511148615351107| 8.533842520592145|
|            9|18.400291333656767| 5.604490502277248|
|           10| 18.56454835768904|4.5114450807098905|
|           11|18.851552824117647| 4.075729557436569|
|           12|19.207714073999153| 4.468694683646846|
|           13| 19.95819012628161| 5.262006204074442|
|           14|20.604332332373676| 4.622973056355365|
|           15| 20.757107834

##Questão 7)
No Spark, existe uma distinção entre transformações (transformations) e ações (actions). Usando como
exemplo os comandos que você utilizou nas Questões 2, 3 e 5, explique essa diferença.

Por que se diz que o
Spark utiliza avaliação preguiçosa (lazy evaluation), e qual é a vantagem prática disso?

RESPOSTA: A diferença é que as transformações de select e filter, não processam os dados imediatamente. Elas apenas definem as etapas que o spark vai executar depois. Já o show e o count, fazem com que o spark realmente execute essas etapas e logo depois apresente o resultado. É por isso que esse funcionamento é chamado de lazy evaluation. O que acontece é que o spark vai acumulando as operações e só começa a processar os dados quando uma ação é chamada. Isso ajuda no desempenho, porque o spark consegue analisar as etapas antes de executar e otimizar o processo.

##Questão 8)
Considerando apenas as corridas em que total_amount seja maior que zero, crie uma nova coluna chamada
percentual_gorjeta, calculada como (tip_amount / total_amount) * 100. Em seguida, exiba as 10
corridas com maior percentual_gorjeta, mostrando as colunas VendorID, total_amount, tip_amount e
percentual_gorjeta.

In [12]:
coluna_vendor = "Vendor ID" if "Vendor ID" in df.columns else "VendorID"

df_corrida_maior_0= df.filter(F.col("total_amount") > 0) \
    .withColumn("percentual_gorjeta", (F.col("tip_amount") / F.col("total_amount")) * 100) \
    .select(coluna_vendor, "total_amount", "tip_amount", "percentual_gorjeta") \
    .orderBy(F.desc("percentual_gorjeta"))

df_corrida_maior_0.show(10)

+--------+------------+----------+------------------+
|VendorID|total_amount|tip_amount|percentual_gorjeta|
+--------+------------+----------+------------------+
|       2|        1.63|      5.27| 323.3128834355828|
|       2|        2.07|      3.68| 177.7777777777778|
|       2|         1.6|      2.82|176.24999999999997|
|       2|        2.33|      3.72|159.65665236051504|
|       2|        2.54|      3.76|148.03149606299212|
|       2|        3.76|      3.96|105.31914893617022|
|       2|        39.7|      40.0|100.75566750629723|
|       2|        0.08|      0.08|             100.0|
|       1|       197.0|     196.0| 99.49238578680203|
|       1|       150.0|     149.0| 99.33333333333333|
+--------+------------+----------+------------------+
only showing top 10 rows


##Questão 9)
A NYC TLC disponibiliza uma tabela de referência que traduz os códigos de zona usados em
PULocationID/DOLocationID para o nome do bairro (Borough) e da zona (Zone) correspondente.

Baixe essa
tabela e carregue em um novo DataFrame:



!wget -q
https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main
/taxi_zone_lookup.csv

zonas = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

zonas.show(5)



Em seguida:

a) Faça o join entre df e zonas, relacionando df.PULocationID com zonas.LocationID, para descobrir o
bairro (Borough) de onde cada corrida partiu;

b) Agrupe o resultado por Borough e conte quantas corridas tiveram origem em cada um;

c) Exiba o resultado ordenado do bairro com mais corridas para o com menos.

In [13]:
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/taxi_zone_lookup.csv

zonas = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

zonas.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [14]:
df_com_mais_corridas = df.join(zonas, df["PULocationID"] == zonas["LocationID"], how="inner") \
    .groupBy("Borough") \
    .agg(F.count("*").alias("total_corridas")) \
    .orderBy(F.desc("total_corridas"))

df_com_mais_corridas.show()

+-------------+--------------+
|      Borough|total_corridas|
+-------------+--------------+
|    Manhattan|       3641752|
|       Queens|        388736|
|     Brooklyn|         60200|
|        Bronx|         12702|
|      Unknown|         12172|
|          N/A|          2421|
|          EWR|           565|
|Staten Island|           195|
+-------------+--------------+



## Questão 10)
Compare o tempo de execução do count() da Questão 1 (sem nenhum agrupamento) com o tempo de
execução do groupBy() da Questão 5.

Baseando-se no conceito de shuffle, explique por que operações de
agrupamento tendem a ser mais custosas do que operações de filtragem ou seleção de colunas, mesmo
processando o mesmo volume de dados.

RESPOSTA: O count é mais rápido porque ele apenas conta as linhas em cada parte do arquivo, sem precisar movimentar muitos dados entre as máquinas. Já o groupby precisa de mais processamento, porqueo spark localiza e agrupa os mesmos valores. Essa movimentação é chamada de shuffle. Decido a esse processo envolver a transferência de uma grande quantidade de dados e poder gerar arquivos temporários, operações de agrupamento normalmente são mais demoradas  e custosas do que operações de contagem ou filtragem dos dados que são consideradas mais simples.
